# Включение масок в откалиброванные научные изображения

Существует три способа определить, какие пиксели в изображении CCD могут нуждаться в маскировании (это дополнительно к любой маске или битовым полям, которые обсерватория, на которой вы снимаете изображения, может предоставить).

Два из них одинаковы для всех научных изображений:

+ Горячие пиксели, которые вряд ли будут правильно откалиброваны путём вычитания тёмного тока, обсуждаемые в [Идентификация горячих пикселей](08-01-Identifying-hot-pixels.ipynb).
+ Плохие пиксели, идентифицированные `ccdproc.ccdmask` из плоских полевых изображений, обсуждаемые в [Создание маски с `ccdmask`](08-02-Creating-a-mask.ipynb).

Третий, идентификация космических лучей, обсуждаемая в [Удаление космических лучей](08-03-Cosmic-ray-removal.ipynb), по своей природе будет разной для каждого научного изображения.

Первые две маски могли бы быть добавлены к научным изображениям во время калибровки научных изображений, если желательно. Они добавляются к научным изображениям здесь как отдельный шаг, потому что во многих ситуациях масскирование в целом нормально и нет особого преимущества в его введении раньше.

Мы начинаем, как обычно, с пары импортов.

In [ ]:
from pathlib import Path

from astropy import units as u
from astropy.nddata import CCDData

import ccdproc as ccdp

## Чтение масок, одинаковых для всех научных изображений

В предыдущих ноутбуках мы построили маску на основе тёмного тока и маску, созданную `ccdmask` из плоского изображения. Отображение сводки информации об уменьшенных изображениях - это удобный способ определить, какие файлы являются масками.

In [ ]:
ex2_path = Path('example2-reduced')

ifc = ccdp.ImageFileCollection(ex2_path)
ifc.summary['file', 'imagetyp']

Мы читаем каждый из них ниже, преобразуя маску в логическую после её чтения.

In [ ]:
mask_ccdmask = CCDData.read(ex2_path / 'mask_from_ccdmask.fits', unit=u.dimensionless_unscaled)
mask_ccdmask.data = mask_ccdmask.data.astype('bool')

mask_hot_pix = CCDData.read(ex2_path / 'mask_from_dark_current.fits', unit=u.dimensionless_unscaled)
mask_hot_pix.data = mask_hot_pix.data.astype('bool')

### Комбинирование масок

Мы комбинируем маски, используя логическое "ИЛИ", поскольку мы хотим замаскировать пиксели, которые плохи по любой причине.

In [ ]:
combined_mask = mask_ccdmask.data | mask_hot_pix.data

Оказывается, мы маскируем примерно 0,056% пикселей на данный момент.

In [ ]:
combined_mask.sum()

## Обнаружение космических лучей

Обнаружение космических лучей было подробно обсуждено в [более раннем разделе](08-03-Cosmic-ray-removal.ipynb). Здесь мы циклируем все откалиброванные научные изображения и:

+ обнаруживаем космические лучи в них,
+ комбинируем маску космических лучей с маской, которая применяется ко всем изображениям,
+ устанавливаем маску изображения на общую маску и
+ сохраняем изображение, перезаписывая откалиброванное научное изображение без маски.

Поскольку обнаружение космических лучей требует некоторого времени, сообщение о статусе отображается перед обработкой каждого изображения.

In [ ]:
ifc.files_filtered()
for ccd, file_name in ifc.ccds(imagetyp='light', return_fname=True):
    print('Working on file {}'.format(file_name))
    new_ccd = ccdp.cosmicray_lacosmic(ccd, readnoise=10, sigclip=8, verbose=True)
    overall_mask = new_ccd.mask | combined_mask
    # If there was already a mask, keep it.
    if ccd.mask is not None:
        ccd.mask = ccd.mask | overall_mask
    else:
        ccd.mask = overall_mask
    # Files can be overwritten only with an explicit option
    ccd.write(ifc.location / file_name, overwrite=True)